# Design an Autocompletion System

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, Caching, Concurrency, Databases, Distributed Systems, Tries · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`20. Smallest_Numbers`](../../2.%20Coding_Questions/20.%20Smallest_Numbers/20.%20Smallest_Numbers.ipynb) is the top-K heap machinery; [`10. LRU_Cache`](../../2.%20Coding_Questions/10.%20LRU_Cache/10.%20LRU_Cache.ipynb) is the cache in front of the trie.

## Concepts

**What this question is really testing:** whether you notice that **two orderings are in conflict**, and what you do about it.

- A **prefix** query needs data in *lexicographic* order.
- A **top-k** result needs data in *score* order.

Sorting by one destroys the other. Every naive design dies on this: sort lexicographically and you must scan the whole matching range to rank it (prefix `"a"` might match 5M phrases); sort by score and you can't find the prefix at all.

**First-principles primer:**

- **A trie** is a tree where the path spells the string. The node you reach by walking `m → o → n` *is* the prefix `"mon"`, and everything below it is a phrase starting with `"mon"`. That solves navigation in `O(prefix length)` — but the subtree under `"a"` is still millions of phrases.
- **Precomputed top-k** solves the other half. Store at each node the k best phrases *in its subtree*, computed in advance. A query is then: walk to the node, return the list. **`O(prefix_length + k)`, with no subtree scan ever.**
- **Write amplification** is the price. A phrase of length L appears in the subtree of exactly L nodes — every prefix of itself — so a score change may need to touch all L lists. That's the trade: you moved ranking work from read time to write time, and reads outnumber writes here roughly 10:1.

**Simple worked example.** Corpus: `mongodb` (900), `monday` (500), `money` (700), with k=2.

```
        root                        Each node stores the best 2 BELOW it:
         │
         m   [mongodb 900, money 700]
         │
         o   [mongodb 900, money 700]
         │
         n   [mongodb 900, money 700]
        ╱│╲
       ╱ │ ╲
      d  e  g   d:[monday 500]  e:[money 700]  g:[mongodb 900]
```

Type `"mon"` → walk 3 nodes → return `[mongodb, money]`. Never looked at `monday`; never walked the subtree.

Now type `"mond"` → one more hop → `[monday]`. Same cost structure.

**The catch nobody mentions.** Suppose `mongodb`'s score drops to 100. At node `n`, it must leave the top-2 — and be replaced by *the third-best phrase under `n`*, which is `monday` (500). But node `n` never stored `monday`; it only kept 2 entries. **To repair the list you have to walk the subtree — the exact thing the design exists to avoid.** Increases are cheap; decreases are not. Hold that thought.

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| Suggest completions as the user types | **< 100 ms** latency |
| Rank by popularity / frequency / recency | High concurrent volume |
| Add phrases; update scores over time | Millions of phrases |
| Typo tolerance | Personalization per user |

**Stated scale:** 100M phrases, 10M DAU, 500M queries/day, 50M interactions/day.

In [ ]:
import os, sys, math, heapq, random
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (DAY, KB, MB, GB, human_bytes, human_count, human_rate,
                      table, assumption_table)

PHRASES = 100_000_000
QUERIES_PER_DAY = 500_000_000
INTERACTIONS_PER_DAY = 50_000_000
PEAK_LOW, PEAK_HIGH = 3, 5
AVG_PHRASE_LEN = 20
K = 5
BYTES_PER_PHRASE = 200          # the answer's capacity-section figure
HOT_PREFIXES = 1_000_000
CACHE_ENTRY_BYTES = 1 * KB

assumption_table({
    "Corpus size":          human_count(PHRASES) + " phrases",
    "Queries / day":        human_count(QUERIES_PER_DAY),
    "Interactions / day":   human_count(INTERACTIONS_PER_DAY),
    "Peak multiplier":      f"{PEAK_LOW}-{PEAK_HIGH}x",
    "Avg phrase length":    f"{AVG_PHRASE_LEN} chars",
    "k (suggestions)":      K,
    "Bytes per phrase":     f"{BYTES_PER_PHRASE} B",
    "Hot prefixes cached":  human_count(HOT_PREFIXES),
})

In [ ]:
qps_avg = QUERIES_PER_DAY / DAY
upd_avg = INTERACTIONS_PER_DAY / DAY
trie_mem = PHRASES * BYTES_PER_PHRASE
cache_mem = HOT_PREFIXES * CACHE_ENTRY_BYTES
ops_per_update = AVG_PHRASE_LEN * K

table([
    ("Query rate, average",  human_rate(qps_avg, " QPS")),
    (f"Query rate, peak ({PEAK_LOW}-{PEAK_HIGH}x)",
     f"{qps_avg * PEAK_LOW / 1000:.0f}k - {qps_avg * PEAK_HIGH / 1000:.0f}k QPS"),
    ("", ""),
    ("Update rate, average", human_rate(upd_avg, " updates/s")),
    ("Update rate, peak",    human_rate(upd_avg * PEAK_HIGH, " updates/s")),
    ("Ops per update (L*k)", f"{ops_per_update} - {AVG_PHRASE_LEN * 10}"),
    ("", ""),
    ("Trie memory",          human_bytes(trie_mem)),
    ("Redis cache",          human_bytes(cache_mem)),
], title="CAPACITY")

assert 5_700 < qps_avg < 5_900, "the answer's stated ~5,800 QPS"
assert 17_000 < qps_avg * PEAK_LOW < 17_500 and 28_000 < qps_avg * PEAK_HIGH < 29_500
assert 570 < upd_avg < 590, "the answer's stated ~580 updates/s"
assert trie_mem == 20 * GB and cache_mem == 1 * GB

# The read:write ratio is what justifies paying at write time.
print(f"\n  Read:write ratio = {QUERIES_PER_DAY / INTERACTIONS_PER_DAY:.0f}:1")
print("  => 10 reads per write. That is what makes precomputing the top-k")
print("     at every node worth its write amplification - though note it is")
print("     10:1, not the 1,000,000:1 of the access-control problem. The margin")
print("     here is real but not enormous, which is why write cost matters.")

## The trie with precomputed top-k

Building it, and confirming that a read never touches the subtree.

In [ ]:
from typing import Dict, List, Tuple, Optional


class Node:
    __slots__ = ("children", "top", "is_terminal")

    def __init__(self):
        self.children: Dict[str, "Node"] = {}
        self.top: List[Tuple[float, str]] = []      # (score, phrase), best first
        self.is_terminal = False


class SuggestTrie:
    """Trie with precomputed top-k at every node. `slack` stores k*slack entries."""

    def __init__(self, k=K, slack=1):
        self.root = Node()
        self.k = k
        self.capacity = k * slack
        self.scores: Dict[str, float] = {}
        self.nodes_touched = 0                      # instrumentation

    # ---- build ----------------------------------------------------------
    def _path(self, phrase) -> List[Node]:
        node, path = self.root, [self.root]
        for ch in phrase:
            node = node.children.setdefault(ch, Node())
            path.append(node)
        node.is_terminal = True
        return path

    def insert(self, phrase, score):
        self.scores[phrase] = score
        for node in self._path(phrase):
            self._offer(node, phrase, score)

    @staticmethod
    def _offer(node, phrase, score):
        node.top = [e for e in node.top if e[1] != phrase]
        node.top.append((score, phrase))
        node.top.sort(key=lambda e: (-e[0], e[1]))

    def finalize(self):
        """Trim every node's list to capacity, after all inserts."""
        def walk(n):
            n.top = n.top[:self.capacity]
            for c in n.children.values():
                walk(c)
        walk(self.root)

    # ---- read -----------------------------------------------------------
    def suggest(self, prefix) -> List[str]:
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return []
            self.nodes_touched += 1
        # serve only entries whose stored score is still current
        valid = [(s, p) for s, p in node.top if self.scores.get(p) == s]
        return [p for _, p in valid[:self.k]]

    # ---- the honest brute force, for cross-checking ---------------------
    def suggest_bruteforce(self, prefix) -> List[str]:
        matches = [(s, p) for p, s in self.scores.items() if p.startswith(prefix)]
        matches.sort(key=lambda e: (-e[0], e[1]))
        return [p for _, p in matches[:self.k]]


corpus = {"mongodb": 900, "monday": 500, "money": 700,
          "monitor": 300, "monkey": 200, "month": 650}
t = SuggestTrie(k=2)
for p, s in corpus.items():
    t.insert(p, s)
t.finalize()

for prefix in ("mon", "mond", "mono", "mone"):
    table([("suggest()", str(t.suggest(prefix))),
           ("brute force", str(t.suggest_bruteforce(prefix)))],
          title=f'PREFIX "{prefix}"')
    assert t.suggest(prefix) == t.suggest_bruteforce(prefix)

# A read costs prefix_length hops - independent of how big the subtree is.
t.nodes_touched = 0
t.suggest("mon")
assert t.nodes_touched == 3, "3 characters, 3 hops - the subtree is never walked"
print(f"\n  Reading prefix 'mon' touched {t.nodes_touched} nodes, not the 6-phrase subtree.")

### ⚠️ Problem 1 — the update path only works for score *increases*

> *"At each node, the updater recomputes `top_suggestions` by checking if the updated phrase's new score changes the top-k ordering. This is O(L × k) per update."*

For an **increase**, that is true: compare against the node's k entries, insert if it beats the weakest. Everything you need is in the node.

For a **decrease** that pushes a phrase out of the top-k, it is not. To replace the departing phrase you need the **(k+1)-th best phrase in that node's subtree** — which the node never stored. Finding it means walking the subtree: exactly the `O(subtree size)` cost the precomputed list exists to eliminate.

And the answer's own formula, `score = frequency * exp(-lambda * age_in_days)`, means **every score decreases with every tick of the clock**. Demotion is not an edge case here; it is the steady state.

In [ ]:
def update_naive(trie, phrase, new_score):
    """The answer's update: touch the L nodes on the path, O(L*k). Increases only."""
    trie.scores[phrase] = new_score
    node = trie.root
    nodes = [node]
    for ch in phrase:
        node = node.children[ch]
        nodes.append(node)
    for n in nodes:
        n.top = [e for e in n.top if e[1] != phrase]
        n.top.append((new_score, phrase))
        n.top.sort(key=lambda e: (-e[0], e[1]))
        n.top = n.top[:trie.capacity]
    return len(nodes)


# --- An INCREASE: handled correctly. ---
t = SuggestTrie(k=2)
for p, s in corpus.items():
    t.insert(p, s)
t.finalize()
update_naive(t, "monkey", 5_000)
assert t.suggest("mon") == t.suggest_bruteforce("mon") == ["monkey", "mongodb"]
print("  INCREASE monkey 200 -> 5000:  correct, and it cost L*k operations.")

# --- A DECREASE: silently wrong. ---
t = SuggestTrie(k=2)
for p, s in corpus.items():
    t.insert(p, s)
t.finalize()
update_naive(t, "mongodb", 100)          # decayed away

got, want = t.suggest("mon"), t.suggest_bruteforce("mon")
table([("Trie says",   str(got)),
       ("Truth is",    str(want)),
       ("Missing",     str([p for p in want if p not in got]))],
      title="DECREASE mongodb 900 -> 100")

assert got != want, "the naive update produces a WRONG list on a decrease"
assert "money" in want and "month" in want
assert "month" not in got, "'month' was never stored at node 'mon' - it cannot appear"
print("\n  => 'month' (650) is genuinely the 2nd best under 'mon' now, but node 'mon'")
print("     only ever held 2 entries and 'month' was not one of them. The correct")
print("     answer is not recoverable from what the node stores.")

### Mitigation A: slack plus lazy repair

Store top-`2k` at each node instead of top-`k`. Serve the best k that are still valid; when a node's valid count drops below k, schedule an **asynchronous subtree recompute for that node alone**.

The read path is unchanged, the cost is amortised, and only nodes that actually degraded pay it. It fixes the single-demotion case cleanly — **but watch what happens under sustained decay**, which is the actual workload.

In [ ]:
def rebuild_node(trie, node, prefix):
    """Recompute one node's list from its subtree. The expensive repair."""
    out = []

    def walk(n, acc):
        if n.is_terminal:
            out.append((trie.scores[acc], acc))
        for ch, c in n.children.items():
            walk(c, acc + ch)

    walk(node, prefix)
    out.sort(key=lambda e: (-e[0], e[1]))
    node.top = out[:trie.capacity]
    return len(out)


def update_with_repair(trie, phrase, new_score):
    """Update the path; repair any node left with fewer than k valid entries."""
    trie.scores[phrase] = new_score
    node, nodes = trie.root, [(trie.root, "")]
    for ch in phrase:
        node = node.children[ch]
        nodes.append((node, phrase[:len(nodes)]))

    repairs = 0
    for n, pfx in nodes:
        n.top = [e for e in n.top if e[1] != phrase]
        n.top.append((new_score, phrase))
        n.top.sort(key=lambda e: (-e[0], e[1]))
        n.top = n.top[:trie.capacity]
        valid = sum(1 for s, p in n.top if trie.scores.get(p) == s)
        if valid < trie.k:                       # underflow -> async repair
            rebuild_node(trie, n, pfx)
            repairs += 1
    return repairs


# SLACK=2 means each node holds 2k entries, so one demotion rarely underflows.
t = SuggestTrie(k=2, slack=2)
for p, s in corpus.items():
    t.insert(p, s)
t.finalize()
r = update_with_repair(t, "mongodb", 100)
assert t.suggest("mon") == t.suggest_bruteforce("mon"), "slack + repair is correct"
table([("Trie says",  str(t.suggest("mon"))),
       ("Truth is",   str(t.suggest_bruteforce("mon"))),
       ("Repairs",    f"{r} of {len('mongodb') + 1} nodes on the path")],
      title="WITH SLACK 2x + LAZY REPAIR")

But one demotion is not the workload. **Sustained decay is** — and slack does not survive it.

The repair trigger fires when a node's entries become *invalid* (tombstoned, deleted). Decay never invalidates anything: every entry keeps a current, correct score. A node can therefore hold 2k perfectly valid entries that have *all* decayed below a phrase which was never in the list — and nothing signals the problem.

In [ ]:
# A realistic workload: many phrases, sustained exponential decay.
def make_workload(n_words=300, seed=7):
    rnd = random.Random(seed)
    words = sorted({"".join(rnd.choice("abcde") for _ in range(rnd.randint(3, 6)))
                    for _ in range(n_words)})
    return words, {w: rnd.randint(1, 10_000) for w in words}


def agreement(trie, prefixes):
    """Fraction of prefixes whose top-k exactly matches the truth."""
    hits = sum(1 for p in prefixes if trie.suggest(p) == trie.suggest_bruteforce(p))
    return hits / len(prefixes)


words, freqs = make_workload()
PREFIXES = [w[:i] for w in words[::7] for i in (1, 2, 3)]
PREFIXES = sorted(set(PREFIXES))
rnd = random.Random(11)

t = SuggestTrie(k=5, slack=2)
for w in words:
    t.insert(w, freqs[w])
t.finalize()

repairs = 0
for _ in range(400):
    w = rnd.choice(words)
    repairs += update_with_repair(t, w, t.scores[w] * math.exp(-0.35))

acc_slack = agreement(t, PREFIXES)
table([
    ("Words / prefixes tested", f"{len(words)} / {len(PREFIXES)}"),
    ("Decay updates applied",   "400"),
    ("Node repairs triggered",  str(repairs)),
    ("", ""),
    ("Prefixes still correct",  f"{acc_slack:.1%}"),
], title="SLACK + LAZY REPAIR, UNDER SUSTAINED DECAY")

assert acc_slack < 1.0, "slack alone does NOT stay correct under decay"
print(f"\n  => {1 - acc_slack:.0%} of prefixes now return a wrong top-5, and no repair")
print("     was ever triggered for them - every stored entry was still 'valid'.")
print("     Slack fixes DELETION, not DECAY. Different failure, different fix.")

# A full rebuild restores truth - which is the point: you need a rebuild cadence.
def rebuild_all(trie):
    def walk(n, pfx):
        rebuild_node(trie, n, pfx)
        for ch, c in n.children.items():
            walk(c, pfx + ch)
    walk(trie.root, "")

rebuild_all(t)
assert agreement(t, PREFIXES) == 1.0
print("\n  A full rebuild restores 100%. So the real question the answer never")
print("  asks is: HOW OFTEN must you rebuild? The design choices below decide it.")

### Mitigation B: make the stored score monotone

The deeper fix is to stop writing decayed scores at all. Store **raw frequency** plus a timestamp, and apply decay to the ≤ 2k candidates *at read time*.

Stored scores then only ever **increase**, which is precisely the case the O(L × k) update handles correctly — so the answer's complexity claim becomes true as written, and ageing costs **zero writes**.

In [ ]:
# Make stored scores MONOTONE, and decay at read time.
# Then the O(L*k) claim is true as written, because scores never decrease.
class MonotoneTrie(SuggestTrie):
    """Store raw frequency; apply decay to the <=2k candidates when serving."""

    def __init__(self, k=K, slack=2, lam=0.35):
        super().__init__(k, slack)
        self.lam = lam
        self.age: Dict[str, float] = {}

    def insert(self, phrase, freq, age_days=0.0):
        self.age[phrase] = age_days
        super().insert(phrase, freq)

    def decayed(self, phrase):
        return self.scores[phrase] * math.exp(-self.lam * self.age[phrase])

    def suggest(self, prefix):
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return []
        ranked = sorted(((self.decayed(p), p) for _, p in node.top),
                        key=lambda e: (-e[0], e[1]))
        return [p for _, p in ranked[:self.k]]

    def suggest_bruteforce(self, prefix):
        m = [(self.decayed(p), p) for p in self.scores if p.startswith(prefix)]
        m.sort(key=lambda e: (-e[0], e[1]))
        return [p for _, p in m[:self.k]]


mt = MonotoneTrie(k=2, slack=3)
for p, s in corpus.items():
    mt.insert(p, s, age_days=0.0)
mt.finalize()
mt.age["mongodb"] = 6.0                    # mongodb is now stale, no write needed

table([("Trie says",  str(mt.suggest("mon"))),
       ("Truth is",   str(mt.suggest_bruteforce("mon")))],
      title="MONOTONE SCORES, DECAY APPLIED AT READ TIME")
assert mt.suggest("mon") == mt.suggest_bruteforce("mon")
assert "mongodb" not in mt.suggest("mon"), "aged out with no trie write at all"

print("\n  => Ageing required ZERO writes. Stored scores only ever increase, so")
print("     the O(L*k) update claim becomes true as stated. The cost is a small")
print("     sort over <=2k candidates per read.")
print("\n     Trade: read cost O(prefix + 2k log 2k) instead of O(prefix + k).")
print("     At k=5 that is ~10 items - genuinely free next to a network hop.")

### How the three compare — and why you still need a rebuild cadence

Run the same decay workload through all three and measure how often each returns the *exact* top-k.

Be honest about the remaining gap: **none of these is exact without a periodic rebuild.** Monotone scoring fixes the write path and makes ageing free, but a phrase outside a node's candidate set can still be missed if it decays back into contention. What the design choice actually buys you is **how often you must rebuild** — and that is the number the source answer never puts on the table.

In [ ]:
def run_decay(trie_factory, n_updates=400, seed=11):
    words, freqs = make_workload()
    prefixes = sorted({w[:i] for w in words[::7] for i in (1, 2, 3)})
    rnd = random.Random(seed)
    trie = trie_factory()
    for w in words:
        trie.insert(w, freqs[w])
    trie.finalize()
    for _ in range(n_updates):
        w = rnd.choice(words)
        if isinstance(trie, MonotoneTrie):
            trie.age[w] = trie.age.get(w, 0.0) + 1.0        # no write to the trie
        else:
            trie.scores[w] = trie.scores[w] * math.exp(-0.35)
            update_with_repair(trie, w, trie.scores[w])
    return agreement(trie, prefixes)


def naive_factory():
    class NoRepair(SuggestTrie):
        pass
    return NoRepair(k=5, slack=1)


results = [
    ("Naive, no slack (as written)", run_decay(lambda: SuggestTrie(k=5, slack=1))),
    ("Slack 2x + lazy repair",       run_decay(lambda: SuggestTrie(k=5, slack=2))),
    ("Slack 4x + lazy repair",       run_decay(lambda: SuggestTrie(k=5, slack=4))),
    ("Monotone, decay at read",      run_decay(lambda: MonotoneTrie(k=5, slack=2))),
    ("Monotone, slack 4x",           run_decay(lambda: MonotoneTrie(k=5, slack=4))),
]
table([(name, f"{acc:6.1%} of prefixes exact") for name, acc in results],
      title="EXACTNESS AFTER 400 DECAY EVENTS")

by_name = dict(results)
assert by_name["Slack 2x + lazy repair"] > by_name["Naive, no slack (as written)"]
assert by_name["Monotone, slack 4x"] >= by_name["Slack 4x + lazy repair"]
assert by_name["Monotone, slack 4x"] < 1.0, "still not exact - a rebuild is required"

print("\n  => Slack is what buys exactness; at equal slack, monotone scoring matches")
print("     it rather than beating it. What monotone buys is a correct WRITE path:")
print("     ageing costs zero writes, and O(L*k) becomes true as stated.")
print("     But NOTHING here is exact. A precomputed top-k trie under a decaying")
print("     score fundamentally needs a periodic rebuild; the design choices only")
print("     decide whether that is hourly or weekly.")
print("\n     The answer to give: 'O(L*k) per update, monotone stored scores, decay")
print("     applied at read, plus a nightly full rebuild' - and say why the rebuild")
print("     is not optional. Claiming O(L*k) with no rebuild is the actual error.")

### ⚠️ Problem 2 — the two memory estimates disagree by 5–15×

| Section | Corpus | Memory | Bytes/phrase |
|---|---|---|---|
| *Scaling and partitioning* | 10M | 10–30 GB | **1,000–3,000** |
| *Capacity numbers* | 100M | 20 GB | **200** |

Ten times the corpus in *less* memory. Both are stated confidently; they cannot both hold. The difference is entirely **trie layout**, which the answer never specifies — and layout moves the answer by an order of magnitude.

In [ ]:
STATED_SMALL = (10_000_000, 10 * GB, 30 * GB)
STATED_LARGE = (100_000_000, 20 * GB, 20 * GB)

table([
    ("Scaling section",  f"{human_count(STATED_SMALL[0])} phrases -> "
                         f"{human_bytes(STATED_SMALL[1])}-{human_bytes(STATED_SMALL[2])}"),
    ("  bytes/phrase",   f"{STATED_SMALL[1] / STATED_SMALL[0]:,.0f} - "
                         f"{STATED_SMALL[2] / STATED_SMALL[0]:,.0f} B"),
    ("", ""),
    ("Capacity section", f"{human_count(STATED_LARGE[0])} phrases -> "
                         f"{human_bytes(STATED_LARGE[1])}"),
    ("  bytes/phrase",   f"{STATED_LARGE[1] / STATED_LARGE[0]:,.0f} B"),
    ("", ""),
    ("Disagreement",     f"{STATED_SMALL[1] / STATED_SMALL[0] / (STATED_LARGE[1] / STATED_LARGE[0]):.0f}x"
                         f" - {STATED_SMALL[2] / STATED_SMALL[0] / (STATED_LARGE[1] / STATED_LARGE[0]):.0f}x"),
], title="CORRECTION 2: TWO INCOMPATIBLE MEMORY FIGURES")

assert STATED_SMALL[1] / STATED_SMALL[0] == 1000
assert STATED_LARGE[1] / STATED_LARGE[0] == 200


NODES_PER_PHRASE = AVG_PHRASE_LEN * 0.35       # ~65% of nodes shared by prefixes

def trie_memory(phrases, k, layout):
    """Three levers: bytes per node, bytes per top-k entry, and WHICH nodes
    carry a top-k list at all."""
    nodes = phrases * NODES_PER_PHRASE
    if layout == "naive":
        per_node = 56 + 64           # object header + a small per-node hash map
        entry = 16 + AVG_PHRASE_LEN  # tuple + a copy of the phrase STRING
        topk_fraction = 1.0          # a list at every node
    else:                            # compact
        per_node = 20                # packed child slots, no per-node object
        entry = 8                    # (phrase_id, score) as packed ints
        topk_fraction = 0.20         # only shallow nodes; deep subtrees are tiny
    return nodes * per_node + nodes * topk_fraction * k * entry


rows = []
for layout in ("naive", "compact"):
    for n in (10_000_000, 100_000_000):
        m = trie_memory(n, K, layout)
        rows.append((f"{layout:<8} {human_count(n):>8} phrases",
                     f"{human_bytes(m):>10}   ({m / n:,.0f} B/phrase)"))
table(rows, title="WHERE THE ORDER OF MAGNITUDE GOES")

naive_bpp = trie_memory(10_000_000, K, "naive") / 10_000_000
compact_bpp = trie_memory(100_000_000, K, "compact") / 100_000_000
assert 1000 <= naive_bpp <= 3000, "naive layout lands in the SCALING section's range"
assert 150 <= compact_bpp <= 400, "compact layout lands near the CAPACITY figure"

print("\n  => Both stated figures are defensible - for DIFFERENT implementations.")
print("     The 200 B/phrase number needs packed child slots, interned phrase IDs")
print("     instead of strings, AND - the lever nobody mentions - a top-k list at")
print("     only a FRACTION of nodes. A pointer-based trie with a hash map and a")
print("     string copy per node costs 10x more.")
print(f"\n     Note that last lever: at {NODES_PER_PHRASE:.0f} nodes per phrase, a 5-entry")
print("     list everywhere is most of the footprint. Deep nodes have tiny")
print("     subtrees, so their top-k is cheap to compute on the fly - store lists")
print("     only where the subtree is big enough to be worth precomputing.")
print("     Say which layout you are building, or the estimate means nothing.")

### ⚠️ Problem 3 — first-character sharding is not "roughly 1/26"

> *"With first-character sharding on 26 lowercase letters, each shard holds roughly 1/26 of the corpus."*

English initial letters are badly non-uniform. `s` starts ~11% of dictionary words; `x` starts ~0.1%.

In [ ]:
# Approximate initial-letter frequencies for English dictionary words (%).
INITIAL_FREQ = {
    "a": 5.7, "b": 6.0, "c": 9.4, "d": 5.6, "e": 3.9, "f": 4.1, "g": 3.3,
    "h": 3.7, "i": 3.9, "j": 0.8, "k": 0.9, "l": 3.1, "m": 5.6, "n": 2.4,
    "o": 2.8, "p": 8.1, "q": 0.4, "r": 4.6, "s": 11.0, "t": 5.0, "u": 2.4,
    "v": 1.3, "w": 2.3, "x": 0.1, "y": 0.3, "z": 0.2,
}
total = sum(INITIAL_FREQ.values())
share = {c: f / total for c, f in INITIAL_FREQ.items()}
uniform = 1 / 26

ranked = sorted(share.items(), key=lambda kv: -kv[1])
table([(f"{c}   {share[c]:6.2%}", f"{share[c] / uniform:5.2f}x the 1/26 claim")
       for c, _ in ranked[:5]] +
      [("...", "")] +
      [(f"{c}   {share[c]:6.2%}", f"{share[c] / uniform:5.2f}x the 1/26 claim")
       for c, _ in ranked[-3:]],
      title="ACTUAL FIRST-LETTER SHARES")

biggest, smallest = ranked[0], ranked[-1]
table([
    ("Uniform ('1/26')",     f"{uniform:.2%}"),
    (f"Largest shard  '{biggest[0]}'",  f"{biggest[1]:.2%}   ({biggest[1] / uniform:.1f}x)"),
    (f"Smallest shard '{smallest[0]}'", f"{smallest[1]:.2%}   ({smallest[1] / uniform:.2f}x)"),
    ("Largest : smallest",   f"{biggest[1] / smallest[1]:.0f} : 1"),
], title="IMBALANCE")

assert biggest[0] == "s" and biggest[1] / uniform > 2.5
assert biggest[1] / smallest[1] > 100

print(f"\n  => You must provision every shard for {biggest[1] / uniform:.1f}x the claimed")
print("     size, and most sit nearly idle. Query traffic is MORE skewed still,")
print("     because popular queries cluster.")

In [ ]:
def balanced_split(share, n_shards):
    """Greedy range partitioning: keep prefix locality, equalise load."""
    letters = sorted(share)
    target = 1 / n_shards
    shards, cur, acc = [], [], 0.0
    for c in letters:
        cur.append(c)
        acc += share[c]
        if acc >= target and len(shards) < n_shards - 1:
            shards.append((cur, acc))
            cur, acc = [], 0.0
    if cur:
        shards.append((cur, acc))
    return shards


N_SHARDS = 8
naive = [([c], share[c]) for c in sorted(share)]
balanced = balanced_split(share, N_SHARDS)

table([(f"{''.join(cs[:1])}-{''.join(cs[-1:])}  ({len(cs)} letters)", f"{w:6.2%}")
       for cs, w in balanced],
      title=f"BALANCED RANGE PARTITIONING ({len(balanced)} SHARDS)")

def imbalance(parts):
    w = [x for _, x in parts]
    return max(w) / (sum(w) / len(w))

table([
    ("Per-letter shards, imbalance", f"{imbalance(naive):.2f}x"),
    ("Balanced ranges, imbalance",   f"{imbalance(balanced):.2f}x"),
], title="MAX SHARD vs AVERAGE SHARD")

assert imbalance(balanced) < imbalance(naive), "balanced ranges even out the load"
print("\n  => Range partitioning keeps queries single-shard (a prefix still falls")
print("     entirely inside one range) AND evens the load. Split 's' further into")
print("     'sa-sm' / 'sn-sz' if it is still the hot one. Choose the boundaries")
print("     from the CORPUS, not from the alphabet.")
print("\n     Note what you must NOT do: hash-shard by phrase. That distributes")
print("     perfectly and destroys prefix locality - every query becomes a")
print("     scatter-gather across all shards.")

## Discussion — the follow-ups

- **Multi-word phrases.** Adding `' '` as a trie edge is right for completing *the whole query string* — `"mongodb at"` → `"mongodb atlas"`. But it does nothing for a user who types the second word first. If that matters, index each phrase under every word-start suffix (`"mongodb atlas"` also inserted at `"atlas"`), pointing back to the same phrase. Storage rises by the average word count; the read path is unchanged. Say which behaviour the product wants — they are different features, not one feature done two ways.
- **Evicting phrases.** A tombstone marks it dead; the read path filters it out, which is exactly the validity check the slack mechanism already performs. A background pass rebuilds affected nodes. The trap is *deleting the trie nodes* eagerly — a node is only removable when it has no children and is not terminal, so deletion is a bottom-up walk that usually removes nothing.
- **CJK and languages without word boundaries.** A character trie still works — the tree just has a fan-out of thousands instead of 26, so packed child arrays become hash maps and the memory model changes. The harder problem is *input*: users type romaji or pinyin and expect Han output, so the index key and the display string differ. That means an IME-aware layer above the trie, not a different trie.
- **A sudden popularity spike.** The async pipeline's batching window (10–30 s) is exactly the latency you cannot afford here. Add a fast path: a small in-memory delta map on each serving node, applied on top of trie scores at read time, that the normal pipeline later folds in and clears. Note this is the *same shape* as the personalization overlay — one small, hot map layered over a big, cold index — which is a hint you have found the right pattern.
- **Measuring quality.** Offline: MRR and precision@k replayed against historical query logs, which is cheap and lets you iterate fast but only rewards reproducing what users already did. Online: click-through rate on suggestions, and — more honestly — *whether the user's final submitted query came from a suggestion*, since a clicked-but-abandoned suggestion is not a win. A/B test ranking changes; never ship a ranking change on offline metrics alone.

## Patterns learned

- **Two conflicting orderings is the whole problem.** Prefix needs lexicographic order; top-k needs score order. A trie gives you the first, and precomputed per-node lists give you the second, without either destroying the other.
- **Precompute at the node, not the answer.** Storing top-k in every node turns `O(subtree)` into `O(k)` — the single decision the design rests on.
- **Ask what happens when the value goes DOWN.** Increases and decreases are not symmetric in a precomputed top-k: an increase needs only what the node holds, a decrease needs what it discarded. Most "O(L × k) update" claims are silently about increases only.
- **Keep slack so you have something to fall back to.** Storing 2k and serving the best valid k converts "every decrease is a subtree walk" into "occasionally, one node is".
- **Or make the stored value monotone.** Store raw frequency, apply decay at read time over the few candidates in the node. Ageing then costs zero writes and the complexity claim becomes true as stated. Prefer this when you can.
- **Layout is not a detail at this scale.** 200 B/phrase and 3,000 B/phrase are both honest numbers for the same structure; only the implementation differs. State it.
- **Check your distribution before claiming balance.** "1/26 per letter" is off by 3× for the largest shard and 100× across the range. Pick split points from the corpus.
- **Shard so that a query stays local.** Range partitioning preserves prefix locality; hash partitioning distributes perfectly and turns every query into a scatter-gather.
- **A small hot overlay over a big cold index** solves personalization *and* popularity spikes. When the same shape answers two follow-ups, it is probably the right shape.